# Sistema CRM Bancario: Reglas SI-ENTONCES

**Curso**: Sistemas basados en conocimiento  
**Profesor**: Alexander Barrantes Calderón  
**Estudiante**: Erick Edgardo Salas Chaverri

## Objetivo
Este notebook implementa el motor de inferencia del sistema CRM bancario con **45 reglas de decisión** basadas en lógica SI-ENTONCES (IF-THEN) y frames.

El sistema automatiza decisiones personalizadas de interacción con clientes combinando cinco dimensiones: **RFM, Arquetipos, Customer Journey, Afinidad de Productos y Canales de Contacto**.

In [29]:
import json
import os

# ============================================================================
# TODAS LAS REGLAS (R1-R95): 65 Originales + 30 Nuevas a 4 Variables Mínimas
# ============================================================================

TODAS_LAS_REGLAS = json.loads('''[
  {
    "id": "R1",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_patrimonial",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión premium"
  },
  {
    "id": "R2",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Incentivos para activación de tarjeta digital"
  },
  {
    "id": "R3",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Oferta integral de seguros familiares"
  },
  {
    "id": "R4",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_transaccional",
      "customer-journey-reactivacion",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "SMS de reactivación con promoción básica"
  },
  {
    "id": "R5",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_patrimonial",
      "customer-journey-riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Retención estratégica con gestor dedicado"
  },
  {
    "id": "R6",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Consultoría de crédito comercial personalizada"
  },
  {
    "id": "R7",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_familia_en_expansion",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Bienvenida familiar con cuentas básicas"
  },
  {
    "id": "R8",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma avanzada de inversiones digital"
  },
  {
    "id": "R9",
    "si": [
      "rfm-monetary-alto",
      "frequency_alta",
      "arquetipo_cliente_patrimonial",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer asesoría financiera personalizada"
  },
  {
    "id": "R10",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R11",
    "si": [
      "rfm-monetary-alto",
      "frequency_alta",
      "arquetipo_cliente_patrimonial",
      "customer-journey-madurez",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión avanzados"
  },
  {
    "id": "R12",
    "si": [
      "rfm-monetary-alto",
      "frequency_alta",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Ofrecer seguros familiares integrales"
  },
  {
    "id": "R13",
    "si": [
      "rfm-monetary-alto",
      "frequency_alta",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma premium de inversiones"
  },
  {
    "id": "R14",
    "si": [
      "rfm-monetary-alto",
      "frequency_alta",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_asesor_personal"
    ],
    "entonces": "Líneas de crédito empresarial avanzadas"
  },
  {
    "id": "R15",
    "si": [
      "rfm-monetary-alto",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_patrimonial",
      "customer-journey-riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto directo del gestor de cuenta"
  },
  {
    "id": "R16",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Ofrecer crédito hipotecario o de consumo"
  },
  {
    "id": "R17",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer-journey-madurez",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R18",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_cliente_transaccional",
      "customer-journey-reactivacion",
      "afinidad_cuenta_corriente",
      "canal_email"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R19",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Ofrecer líneas de crédito comercial"
  },
  {
    "id": "R20",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña digital ofreciendo primera tarjeta con beneficios"
  },
  {
    "id": "R21",
    "si": [
      "rfm-monetary-alto",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer-journey-riesgo_de_abandono",
      "afinidad_cuenta_corriente",
      "canal_telefono"
    ],
    "entonces": "Campaña de retención personalizada"
  },
  {
    "id": "R22",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R23",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma digital de inversiones"
  },
  {
    "id": "R24",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-riesgo_de_abandono",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Contacto proactivo del asesor comercial"
  },
  {
    "id": "R25",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_familia_en_expansion",
      "customer-journey-reactivacion",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Campaña automática: Línea de crédito personal con tasa preferencial"
  },
  {
    "id": "R26",
    "si": [
      "rfm-monetary-alto",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_cliente_patrimonial",
      "customer-journey-adquisicion",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Bienvenida ejecutiva personalizada"
  },
  {
    "id": "R27",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente",
      "canal_sucursal"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R28",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_cliente_transaccional",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R29",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-riesgo_de_abandono",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R30",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito",
      "canal_telefono"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R31",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_antiguo",
      "arquetipo_familia_en_expansion",
      "customer-journey-riesgo_de_abandono",
      "afinidad_seguros",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto de retención con asesor"
  },
  {
    "id": "R32",
    "si": [
      "rfm-monetary-bajo",
      "frequency_baja",
      "recency_inactivo",
      "arquetipo_cliente_transaccional",
      "customer-journey-reactivacion",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R33",
    "si": [
      "rfm-monetary-medio",
      "frequency_baja",
      "recency_muy_reciente",
      "arquetipo_familia_en_expansion",
      "customer-journey-adquisicion",
      "afinidad_credito_personal",
      "canal_sucursal"
    ],
    "entonces": "Paquete de bienvenida familiar"
  },
  {
    "id": "R34",
    "si": [
      "rfm-monetary-medio",
      "frequency_alta",
      "recency_reciente",
      "arquetipo_cliente_patrimonial",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Ofrecer productos de inversión intermedios"
  },
  {
    "id": "R35",
    "si": [
      "rfm-monetary-medio",
      "frequency_alta",
      "recency_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Acceso a portafolio de inversiones robo-advisor"
  },
  {
    "id": "R36",
    "si": [
      "rfm-monetary-medio",
      "frequency_media",
      "recency_reciente",
      "arquetipo_familia_en_expansion",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Ofertas de productos complementarios"
  },
  {
    "id": "R37",
    "si": [
      "rfm-monetary-alto",
      "frequency_media",
      "recency_reciente",
      "arquetipo_cliente_patrimonial",
      "customer-journey-madurez",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Revisión anual de portafolio"
  },
  {
    "id": "R38",
    "si": [
      "rfm-monetary-bajo",
      "frequency_media",
      "recency_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Oferta de tarjeta crédito sin anualidad"
  },
  {
    "id": "R39",
    "si": [
      "rfm-monetary-medio",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_cliente_transaccional",
      "customer-journey-madurez",
      "afinidad_cuenta_corriente",
      "canal_sms"
    ],
    "entonces": "Comunicación segmentada con incentivos"
  },
  {
    "id": "R40",
    "si": [
      "rfm-monetary-alto",
      "frequency_media",
      "recency_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_asesor_personal"
    ],
    "entonces": "Soluciones de tesorería avanzada"
  },
  {
    "id": "R41",
    "si": [
      "rfm-monetary-bajo",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Campaña automática: Línea de crédito personal con tasa preferencial"
  },
  {
    "id": "R42",
    "si": [
      "rfm-monetary-alto",
      "frequency_media",
      "recency_muy_reciente",
      "arquetipo_profesional_joven_digital",
      "customer-journey-adquisicion",
      "afinidad_tarjeta_credito",
      "canal_aplicacion_movil"
    ],
    "entonces": "Tarjeta premium con beneficios ejecutivos"
  },
  {
    "id": "R43",
    "si": [
      "rfm-monetary-medio",
      "frequency_media",
      "recency_reciente",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Línea de crédito flexible para negocio"
  },
  {
    "id": "R44",
    "si": [
      "rfm-monetary-bajo",
      "frequency_media",
      "recency_reciente",
      "arquetipo_cliente_transaccional",
      "customer-journey-crecimiento",
      "afinidad_cuenta_corriente",
      "canal_aplicacion_movil"
    ],
    "entonces": "Campaña automática: Cuenta corriente con beneficios básicos"
  },
  {
    "id": "R45",
    "si": [
      "rfm-monetary-medio",
      "frequency_media",
      "recency_antiguo",
      "arquetipo_cliente_patrimonial",
      "customer-journey-riesgo_de_abandono",
      "afinidad_fondos_inversion",
      "canal_asesor_personal"
    ],
    "entonces": "Contacto de reconexión con análisis de valor"
  },
  {
    "id": "R46",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_familia_en_expansion",
      "customer-journey-crecimiento",
      "afinidad_credito_personal",
      "canal_telefono"
    ],
    "entonces": "Crédito familiar con financiamiento flexible"
  },
  {
    "id": "R47",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-crecimiento",
      "afinidad_tarjeta_credito",
      "canal_email"
    ],
    "entonces": "Oferta de tarjeta crédito con cashback digital"
  },
  {
    "id": "R48",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_patrimonial",
      "customer-journey-activacion",
      "afinidad_seguros",
      "canal_sucursal"
    ],
    "entonces": "Consultoría seguros para patrimonio"
  },
  {
    "id": "R49",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-activacion",
      "afinidad_fondos_inversion",
      "canal_aplicacion_movil"
    ],
    "entonces": "Plataforma inversiones para negocios"
  },
  {
    "id": "R50",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_transaccional",
      "customer-journey-crecimiento",
      "afinidad_tarjeta_credito",
      "canal_sucursal"
    ],
    "entonces": "Primera tarjeta de crédito en sucursal"
  },
  {
    "id": "R51",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_familia_en_expansion",
      "customer-journey-reactivacion",
      "afinidad_seguros",
      "canal_telefono"
    ],
    "entonces": "Reactivación con seguros familiares"
  },
  {
    "id": "R52",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_fondos_inversion",
      "canal_email"
    ],
    "entonces": "Newsletter semanal de inversiones premium"
  },
  {
    "id": "R53",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_sms"
    ],
    "entonces": "SMS con ofertas de crédito empresarial"
  },
  {
    "id": "R54",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_transaccional",
      "customer-journey-activacion",
      "afinidad_cuenta_corriente",
      "canal_telefono"
    ],
    "entonces": "Llamada bienvenida para activación"
  },
  {
    "id": "R55",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_patrimonial",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_email"
    ],
    "entonces": "Línea de crédito premium con tasas preferenciales"
  },
  {
    "id": "R56",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_familia_en_expansion",
      "customer-journey-crecimiento",
      "afinidad_cuenta_corriente",
      "canal_email"
    ],
    "entonces": "Cuenta corriente familiar con beneficios"
  },
  {
    "id": "R57",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_tarjeta_credito",
      "canal_sms"
    ],
    "entonces": "Beneficios exclusivos tarjeta crédito digital"
  },
  {
    "id": "R58",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_email"
    ],
    "entonces": "Fondos empresariales para expansión"
  },
  {
    "id": "R59",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_patrimonial",
      "customer-journey-reactivacion",
      "afinidad_fondos_inversion",
      "canal_sms"
    ],
    "entonces": "Reactivación con portafolio básico"
  },
  {
    "id": "R60",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_credito_personal",
      "canal_sms"
    ],
    "entonces": "SMS con ofertas de consolidación de deuda"
  },
  {
    "id": "R61",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_seguros",
      "canal_aplicacion_movil"
    ],
    "entonces": "Seguros de vida digital a través de app"
  },
  {
    "id": "R62",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Seguros comerciales para negocio"
  },
  {
    "id": "R63",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_transaccional",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion",
      "canal_sucursal"
    ],
    "entonces": "Asesoría en sucursal para comenzar inversión"
  },
  {
    "id": "R64",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_seguros",
      "canal_email"
    ],
    "entonces": "Paquete seguros integral familiar premium"
  },
  {
    "id": "R65",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-reactivacion",
      "afinidad_credito_personal",
      "canal_aplicacion_movil"
    ],
    "entonces": "Reactivación con crédito personal digital"
  },
  {
    "id": "R66",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_patrimonial",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Asesoría de inversión personalizada por valor alto"
  },
  {
    "id": "R67",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_profesional_joven_digital",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Activación de tarjeta con beneficios digitales"
  },
  {
    "id": "R68",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_familia_en_expansion",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Bienvenida familiar con cuenta básica"
  },
  {
    "id": "R69",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-crecimiento",
      "afinidad_credito_personal"
    ],
    "entonces": "Línea de crédito para expansión empresarial"
  },
  {
    "id": "R70",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_transaccional",
      "customer-journey-madurez",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Oferta de servicios complementarios"
  },
  {
    "id": "R71",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_patrimonial",
      "customer-journey-riesgo_de_abandono",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Retención con portafolio básico"
  },
  {
    "id": "R72",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_seguros"
    ],
    "entonces": "Paquete seguros premium familiar"
  },
  {
    "id": "R73",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_profesional_joven_digital",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Plataforma inversiones progresiva"
  },
  {
    "id": "R74",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-activacion",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Tarjeta para emprendedor con beneficios básicos"
  },
  {
    "id": "R75",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_transaccional",
      "customer-journey-crecimiento",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Tarjeta premium con beneficios exclusivos"
  },
  {
    "id": "R76",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_familia_en_expansion",
      "customer-journey-activacion",
      "afinidad_credito_personal"
    ],
    "entonces": "Crédito personal familiar inicial"
  },
  {
    "id": "R77",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_seguros"
    ],
    "entonces": "Seguros de vida básicos para profesional"
  },
  {
    "id": "R78",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_patrimonial",
      "customer-journey-madurez",
      "afinidad_credito_personal"
    ],
    "entonces": "Línea de crédito premium patrimonial"
  },
  {
    "id": "R79",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_transaccional",
      "customer-journey-reactivacion",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Reactivación con beneficios transaccionales"
  },
  {
    "id": "R80",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_familia_en_expansion",
      "customer-journey-madurez",
      "afinidad_seguros"
    ],
    "entonces": "Seguros familiares esenciales"
  },
  {
    "id": "R81",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_profesional_joven_digital",
      "customer-journey-adquisicion",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Portafolio inversiones profesional inicial"
  },
  {
    "id": "R82",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-madurez",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Fondos operacionales para negocio"
  },
  {
    "id": "R83",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_transaccional",
      "customer-journey-adquisicion",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Primera tarjeta crédito básica"
  },
  {
    "id": "R84",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_familia_en_expansion",
      "customer-journey-activacion",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Cuenta familiar premium con beneficios"
  },
  {
    "id": "R85",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_profesional_joven_digital",
      "customer-journey-reactivacion",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Reactivación tarjeta con bonificación"
  },
  {
    "id": "R86",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-madurez",
      "afinidad_seguros"
    ],
    "entonces": "Seguros comerciales básicos"
  },
  {
    "id": "R87",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_cliente_transaccional",
      "customer-journey-madurez",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Portafolio inversión para cliente activo"
  },
  {
    "id": "R88",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_patrimonial",
      "customer-journey-reactivacion",
      "afinidad_credito_personal"
    ],
    "entonces": "Línea crédito patrimonial secundaria"
  },
  {
    "id": "R89",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_profesional_joven_digital",
      "customer-journey-crecimiento",
      "afinidad_fondos_inversion"
    ],
    "entonces": "Inicio inversión digital progresiva"
  },
  {
    "id": "R90",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_emprendedor_pequeño_empresario",
      "customer-journey-adquisicion",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Tarjeta empresarial premium inicial"
  },
  {
    "id": "R91",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_familia_en_expansion",
      "customer-journey-crecimiento",
      "afinidad_seguros"
    ],
    "entonces": "Seguros complementarios para familia"
  },
  {
    "id": "R92",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_cliente_patrimonial",
      "customer-journey-adquisicion",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Cuenta patrimonial básica inaugural"
  },
  {
    "id": "R93",
    "si": [
      "rfm-monetary-alto",
      "arquetipo_profesional_joven_digital",
      "customer-journey-madurez",
      "afinidad_tarjeta_credito"
    ],
    "entonces": "Tarjeta premium profesional premium"
  },
  {
    "id": "R94",
    "si": [
      "rfm-monetary-medio",
      "arquetipo_cliente_transaccional",
      "customer-journey-crecimiento",
      "afinidad_credito_personal"
    ],
    "entonces": "Crédito personal para crecimiento transaccional"
  },
  {
    "id": "R95",
    "si": [
      "rfm-monetary-bajo",
      "arquetipo_familia_en_expansion",
      "customer-journey-reactivacion",
      "afinidad_cuenta_corriente"
    ],
    "entonces": "Reactivación familiar con beneficios básicos"
  }
]''')

print("Todas las 95 reglas (R1-R95) cargadas exitosamente")
print(f"Total de reglas: {len(TODAS_LAS_REGLAS)}")


Todas las 95 reglas (R1-R95) cargadas exitosamente
Total de reglas: 95


In [30]:
# ============================================================================
# Caracteristicas UTILIZADOS EN LAS REGLAS SI-ENTONCES
# ============================================================================

# Lista de caracteristicas principales en el campo "si" de las 45 reglas
CARACTERISTICAS = [
    "rfm-monetary",      # Valor monetario del cliente (alto, medio, bajo)
    "frequency",         # Frecuencia de transacciones (alta, media, baja)
    "recency",           # Antigüedad de última transacción (muy_reciente, reciente, antiguo, inactivo)
    "arquetipo",         # Tipo de cliente (cliente_patrimonial, profesional_joven_digital, etc.)
    "customer-journey",  # Etapa en el ciclo de vida (adquisicion, activacion, crecimiento, madurez, etc.)
    "afinidad",          # Afinidad por producto (fondos_inversion, tarjeta_credito, seguros, etc.)
    "canal"              # Canal de contacto preferido (asesor_personal, aplicacion_movil, email, etc.)
]

# Mostrar información de prefijos
print("=" * 70)
print("PREFIJOS UTILIZADOS EN LAS REGLAS CRM")
print("=" * 70)
print(f"\nTotal de prefijos: {len(CARACTERISTICAS)}\n")
for i, prefijo in enumerate(CARACTERISTICAS, 1):
    print(f"{i}. {prefijo}")

print("\n" + "=" * 70)


PREFIJOS UTILIZADOS EN LAS REGLAS CRM

Total de prefijos: 7

1. rfm-monetary
2. frequency
3. recency
4. arquetipo
5. customer-journey
6. afinidad
7. canal



In [31]:
# ============================================================================
# SUMARIO: Cantidad de Hechos en las Reglas
# ============================================================================

import pandas as pd

print("\n" + "=" * 80)
print("SUMARIO: CANTIDAD DE HECHOS (CONDICIONES) EN LAS REGLAS")
print("=" * 80 + "\n")

# Calcular distribución de hechos
distribucion_hechos = {}
for regla in TODAS_LAS_REGLAS:
    num_hechos = len(regla['si'])
    if num_hechos not in distribucion_hechos:
        distribucion_hechos[num_hechos] = []
    distribucion_hechos[num_hechos].append(regla['id'])

# Crear tabla ordenada
tabla_sumario = []
for num_hechos in sorted(distribucion_hechos.keys()):
    cantidad = len(distribucion_hechos[num_hechos])
    reglas_ids = ', '.join(distribucion_hechos[num_hechos])
    especificidad = "MEDIA (fallback)" if num_hechos == 5 else "ALTA (prioritaria)" if num_hechos >= 6 else "BAJA"
    
    tabla_sumario.append({
        'Cantidad de Hechos': num_hechos,
        'Cantidad de Reglas': cantidad,
        'Porcentaje': f'{(cantidad / len(TODAS_LAS_REGLAS) * 100):.1f}%',
        'Especificidad': especificidad,
        'Ejemplos de Reglas': reglas_ids[:50] + '...' if len(reglas_ids) > 50 else reglas_ids
    })

df_sumario = pd.DataFrame(tabla_sumario)
print(df_sumario.to_string(index=False))

print("\n" + "=" * 80)
print(f"\nTOTAL DE REGLAS: {len(TODAS_LAS_REGLAS)}")
print(f"RANGO DE HECHOS: {min(distribucion_hechos.keys())} a {max(distribucion_hechos.keys())} condiciones")

# Estadísticas
promedio_hechos = sum(len(r['si']) for r in TODAS_LAS_REGLAS) / len(TODAS_LAS_REGLAS)
print(f"PROMEDIO DE HECHOS POR REGLA: {promedio_hechos:.2f}")

print("\nDEAGLOSE DETALLADO:")
for num_hechos in sorted(distribucion_hechos.keys()):
    cantidad = len(distribucion_hechos[num_hechos])
    porcentaje = (cantidad / len(TODAS_LAS_REGLAS) * 100)
    barra = "█" * int(porcentaje / 2)
    print(f"  {num_hechos} condiciones: {cantidad:2d} reglas ({porcentaje:5.1f}%) {barra}")

print("\n" + "=" * 80)




SUMARIO: CANTIDAD DE HECHOS (CONDICIONES) EN LAS REGLAS

 Cantidad de Hechos  Cantidad de Reglas Porcentaje      Especificidad                                    Ejemplos de Reglas
                  4                  30      31.6%               BAJA R66, R67, R68, R69, R70, R71, R72, R73, R74, R75, ...
                  5                  28      29.5%   MEDIA (fallback) R1, R2, R3, R4, R5, R6, R7, R8, R46, R47, R48, R49...
                  6                   5       5.3% ALTA (prioritaria)                                R9, R11, R12, R13, R14
                  7                  32      33.7% ALTA (prioritaria) R10, R15, R16, R17, R18, R19, R20, R21, R22, R23, ...


TOTAL DE REGLAS: 95
RANGO DE HECHOS: 4 a 7 condiciones
PROMEDIO DE HECHOS POR REGLA: 5.41

DEAGLOSE DETALLADO:
  4 condiciones: 30 reglas ( 31.6%) ███████████████
  5 condiciones: 28 reglas ( 29.5%) ██████████████
  6 condiciones:  5 reglas (  5.3%) ██
  7 condiciones: 32 reglas ( 33.7%) ████████████████



In [32]:
class SistemaCRM:
    """
    Sistema de recomendación CRM basado en reglas de conocimiento
    Utiliza RFM analysis, customer archetypes y journey stages para recomendar acciones

    ACTUALIZADO: Carga ahora las 95 REGLAS COMPLETAS del sistema (R1-R95)
    - 45 Reglas originales (R1-R45): con 5-7 variables
    - 20 Reglas con 5 variables (R46-R65): especificidad media
    - 30 Reglas con 4 variables (R66-R95): especificidad baja (sin canal)
    
    LÓGICA DE DECISIÓN MEJORADA:
    - Si especificidad ALTA (6-7): Mostrar recomendación + trazabilidad
    - Si especificidad MENOR (5 o 4): Solo mostrar trazabilidad (sin recomendación)
    """

    def __init__(self):
        self.hechos = []
        self.reglas = TODAS_LAS_REGLAS

    def cargar_reglas_embebidas(self):
        """Carga las reglas embebidas en el script (ya están en TODAS_LAS_REGLAS)"""
        try:
            print(f"[OK] Cargadas {len(self.reglas)} reglas desde TODAS_LAS_REGLAS\n")
        except Exception as e:
            print(f"[ERROR] Error al cargar reglas: {e}")

    def agregar_hecho(self, hecho):
        """Agrega un hecho al conocimiento actual"""
        if hecho not in self.hechos:
            self.hechos.append(hecho)

    def agregar_hechos(self, hechos):
        """Agrega múltiples hechos"""
        for hecho in hechos:
            self.agregar_hecho(hecho)

    def limpiar_hechos(self):
        """Limpia todos los hechos"""
        self.hechos = []

    def inferir(self):
        """
        MOTOR DE INFERENCIA: Busca reglas que coincidan con los hechos actuales
        
        ALGORITMO DE RESOLUCIÓN DE CONFLICTOS:
        1. Evalúa todas las reglas contra hechos
        2. Retorna todas las reglas que aplican
        3. Ordena por especificidad (mayor # condiciones = mayor prioridad)
        """
        reglas_coincidentes = []

        for regla in self.reglas:
            # Verificar si todas las condiciones de la regla están en los hechos (lógica AND)
            if all(condicion in self.hechos for condicion in regla['si']):
                reglas_coincidentes.append(regla)

        # Ordenar por especificidad (mayor número de condiciones primero)
        reglas_coincidentes.sort(key=lambda r: len(r['si']), reverse=True)

        return reglas_coincidentes

    def mostrar_resultado(self, cliente_nombre="Cliente"):
        """
        Muestra los resultados de la inferencia con detalles de especificidad.
        
        LÓGICA DE VISIBILIDAD:
        - Si especificidad es ALTA (6-7 variables): Mostrar recomendación + trazabilidad completa
        - Si especificidad es MENOR (5 o 4 variables): Solo mostrar trazabilidad (audit trail)
        """
        print(f"+" + "="*62 + "+")
        print(f"| ANALISIS CRM: {cliente_nombre:50} |")
        print(f"+" + "="*62 + "+\n")

        print(f"[HECHOS] HECHOS IDENTIFICADOS ({len(self.hechos)} condiciones):")
        for hecho in self.hechos:
            print(f"   - {hecho}")

        print(f"\n[BUSCANDO] BUSCANDO REGLAS APLICABLES...\n")

        reglas_aplicables = self.inferir()

        if reglas_aplicables:
            print(f"[ENCONTRADAS] Se encontraron {len(reglas_aplicables)} regla(s):\n")

            # Obtener regla de máxima especificidad
            regla_optima = reglas_aplicables[0]
            n_cond = len(regla_optima['si'])
            especificidad_alta = n_cond >= 6  # ALTA si tiene 6-7 variables

            if especificidad_alta:
                # MODO: RECOMENDACIÓN COMPLETA (cuando especificidad es ALTA)
                spec_label = "ALTA (6-7)"

                print(f"{'*' * 65}")
                print(f"[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)")
                print(f"{'*' * 65}")
                print(f"Regla ID: {regla_optima['id']}")
                print(f"Especificidad: {spec_label} ({n_cond} condiciones)")
                print(f"\n[ACCION] ACCION RECOMENDADA:\n  >> {regla_optima['entonces']}")
                print(f"{'*' * 65}\n")
            else:
                # MODO: SOLO TRAZABILIDAD (cuando especificidad es MEDIA o BAJA)
                spec_label = "MEDIA (5)" if n_cond == 5 else "BAJA (4)"
                print(f"[NOTA] NOTA: Especificidad {spec_label} ({n_cond} variables)")
                print(f"Se muestran las reglas usadas por trazabilidad (sin recomendacion).\n")

            # TRAZABILIDAD: Mostrar TODAS las reglas aplicables (audit trail)
            print(f"{'─' * 65}")
            print(f"[TRAZABILIDAD] TRAZABILIDAD: Reglas Aplicables ({len(reglas_aplicables)} total):")
            print(f"{'─' * 65}\n")

            for idx, regla in enumerate(reglas_aplicables, 1):
                n_cond_r = len(regla['si'])
                spec_r = "ALTA (6-7)" if n_cond_r >= 6 else "MEDIA (5)" if n_cond_r == 5 else "BAJA (4)"
                
                # Marcar la regla óptima
                marca = "[OPTIMA]" if idx == 1 else ""
                
                print(f"{idx}. {regla['id']} {marca}")
                print(f"   Especificidad: {spec_r} ({n_cond_r} condiciones)")
                print(f"   Condiciones: {', '.join(regla['si'][:3])}{'...' if len(regla['si']) > 3 else ''}")
                print(f"   Accion: {regla['entonces']}")
                print()

            print(f"{'─' * 65}\n")
        else:
            print(f"[NO ENCONTRADAS] No se encontraron reglas que coincidan con estos hechos")

        print(f"\n{'=' * 65}\n")
        return reglas_aplicables


# Crear instancia del sistema con 95 reglas
sistema = SistemaCRM()

print("="*65)
print("Sistema CRM inicializado con 95 reglas completas (R1-R95)")
print("="*65)

Sistema CRM inicializado con 95 reglas completas (R1-R95)


In [33]:
# ============================================================================
# SISTEMA CRM: Sistema único con 95 reglas totales (R1-R95)
# ============================================================================

class SistemaCRMExtendido(SistemaCRM):
    """
    Sistema CRM con 95 reglas totales en una sola colección
    - 45 Reglas originales (R1-R45): con 5-7 variables
    - 20 Nuevas reglas (R46-R65): con 5 variables mínimas
    - 30 Nuevas reglas (R66-R95): con 4 variables (sin canal)
    Total: 95 reglas en TODAS_LAS_REGLAS
    """
    
    def __init__(self):
        self.hechos = []
        self.reglas = TODAS_LAS_REGLAS
        
    def cargar_reglas(self):
        """Informa sobre las reglas cargadas"""
        print(f"[OK] Sistema CRM cargado con {len(self.reglas)} reglas (R1-R95)")


# Crear instancia del sistema
sistema_extendido = SistemaCRMExtendido()
sistema_extendido.cargar_reglas()

print("=" * 70)
print("Sistema CRM inicializado con 95 reglas totales (R1-R95)")
print("=" * 70)


[OK] Sistema CRM cargado con 95 reglas (R1-R95)
Sistema CRM inicializado con 95 reglas totales (R1-R95)


## Representación de Clientes con Frames

En esta sección, implementamos un sistema de **Frames con Herencia** para representar clientes del banco.
Esto permite:
- Organizar información estructurada de clientes
- Heredar propiedades comunes (Persona → Cliente específico)
- Aplicar reglas CRM basadas en atributos del frame
- Demostrar conocimiento orientado a objetos

**Arquitectura de Frames:**
```
Persona (Frame Padre)
  ├─ nombre
  ├─ apellido
  ├─ edad
  └─ dirección

Cliente (hereda de Persona)
  ├─ numero_cuenta
  ├─ saldo
  ├─ rfm-monetary
  ├─ frequency
  ├─ recency
  ├─ customer-journey
  ├─ afinidad_producto
  └─ canal_contacto
```

In [34]:
from typing import Optional, Dict, Any

# ============================================================================
# CLASE FRAME: Representación de Conocimiento Estructurado con Herencia
# ============================================================================

class Frame:
    """
    Clase para representar un Frame (marco/esquema) con soporte para herencia.

    Los frames permiten organizar información sobre entidades del mundo real,
    con la capacidad de heredar atributos de frames padres.

    Atributos:
    - nombre: Identificador único del frame
    - slots: Diccionario de atributos (clave-valor)
    - padre: Frame padre del cual hereda valores
    - arquetipo: Tipo de cliente (para fines CRM)
    """

    def __init__(self, nombre: str, slots: Optional[Dict[str, Any]] = None,
                 padre: Optional['Frame'] = None, arquetipo: str = ""):
        self.nombre = nombre
        self.slots = slots if slots else {}
        self.padre = padre
        self.arquetipo = arquetipo

    def set_slot(self, atributo: str, valor: Any) -> None:
        """Asigna un valor a un slot del frame"""
        self.slots[atributo] = valor

    def get_slot(self, atributo: str) -> Any:
        """
        Obtiene el valor de un slot.
        Si no existe en el frame actual, busca en el frame padre (HERENCIA).
        """
        if atributo in self.slots:
            return self.slots[atributo]
        if self.padre:
            return self.padre.get_slot(atributo)
        return None

    def get_all_slots(self) -> Dict[str, Any]:
        """Obtiene todos los slots (propios + heredados)"""
        resultado = {}
        if self.padre:
            resultado.update(self.padre.get_all_slots())
        resultado.update(self.slots)
        return resultado

    def get_hechos_crm(self) -> list:
        """
        Convierte los slots del frame en hechos para el motor de inferencia CRM.
        ACTUALIZADO: Mapea correctamente los nombres de slots a condiciones de reglas.
        """
        hechos = []
        slots = self.get_all_slots()

        # Mapeo de slots a prefijos de hechos (para coincidencia con reglas)
        slot_to_fact_prefix = {
            'afinidad_producto': 'afinidad',      # afinidad_producto -> afinidad_*
            'canal_contacto': 'canal',            # canal_contacto -> canal_*
        }

        for atributo, valor in slots.items():
            if isinstance(valor, str) and valor not in ["No definido", "No definida"]:
                # Obtener el prefijo correcto para el hecho
                prefijo = slot_to_fact_prefix.get(atributo, atributo)

                # Formato: prefijo_valor (ej: rfm-monetary-alto, afinidad_fondos_inversion)
                hecho = f"{prefijo}_{valor}".lower().replace(" ", "_")
                hechos.append(hecho)
            elif isinstance(valor, bool) and valor:
                hechos.append(f"{atributo}_si")

        return hechos

    def __repr__(self) -> str:
        return f"Frame({self.nombre}) [{self.arquetipo}]"

    def __str__(self) -> str:
        slots_str = ", ".join([f"{k}={v}" for k, v in self.slots.items()])
        return f"{self.nombre}: {slots_str}"


print("Clase Frame actualizada con mapeo correcto de slots a hechos CRM")


Clase Frame actualizada con mapeo correcto de slots a hechos CRM


### Creación del Frame Padre: Persona

Definimos el frame base **Persona** con atributos comunes a todos los individuos.

In [35]:
# ============================================================================
# FRAME PADRE: Persona
# ============================================================================

persona = Frame(
    nombre="Persona",
    slots={
        "nombre": "No definido",
        "apellido": "No definido",
        "edad": 0,
        "direccion": "No definida"
    },
    padre=None
)

print("Frame Padre 'Persona' creado con atributos base")
print(f"  - nombre")
print(f"  - apellido")
print(f"  - edad")
print(f"  - dirección")


Frame Padre 'Persona' creado con atributos base
  - nombre
  - apellido
  - edad
  - dirección


### Creación de Frames de Clientes Específicos

Creamos diferentes clientes (frames hijos) que heredan de Persona.
Cada cliente tiene atributos RFM y propiedades CRM específicas.

In [36]:
# ============================================================================
# FRAMES DE CLIENTES: Heredan de Persona
# ============================================================================

# Cliente 1: Patrimonial - Alto Valor (ACTUALIZADO: includes frequency_media y customer-journey-crecimiento)
cliente_patrimonial = Frame(
    nombre="Cliente_Roberto",
    slots={
        "nombre": "Roberto",
        "apellido": "Martínez",
        "edad": 55,
        "direccion": "Escalante, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-001",
        "arquetipo": "cliente_patrimonial",
        "rfm-monetary": "alto",
        "frequency": "media",
        "recency": "reciente",
        "customer-journey": "crecimiento",
        "afinidad_producto": "fondos_inversion",
        "canal_contacto": "asesor_personal",
        "saldo": 500000.00
    },
    padre=persona,
    arquetipo="cliente_patrimonial"
)

# Cliente 2: Profesional Joven Digital
cliente_digital = Frame(
    nombre="Cliente_Andrea",
    slots={
        "nombre": "Andrea",
        "apellido": "González",
        "edad": 28,
        "direccion": "Barrio Escalante, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-002",
        "arquetipo": "profesional_joven_digital",
        "rfm-monetary": "bajo",
        "frequency": "baja",
        "recency": "muy_reciente",
        "customer-journey": "activacion",
        "afinidad_producto": "tarjeta_credito",
        "canal_contacto": "aplicacion_movil",
        "saldo": 15000.00
    },
    padre=persona,
    arquetipo="profesional_joven_digital"
)

# Cliente 3: Familia en Expansión (ACTUALIZADO: includes afinidad_seguros y canal_email)
cliente_familia = Frame(
    nombre="Cliente_familia_Lopez",
    slots={
        "nombre": "Carlos",
        "apellido": "López",
        "edad": 40,
        "direccion": "Moravia, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-003",
        "arquetipo": "familia_en_expansion",
        "rfm-monetary": "medio",
        "frequency": "baja",
        "recency": "muy_reciente",
        "customer-journey": "adquisicion",
        "afinidad_producto": "seguros",
        "canal_contacto": "email",
        "saldo": 50000.00
    },
    padre=persona,
    arquetipo="familia_en_expansion"
)

# Cliente 4: Emprendedor
cliente_emprendedor = Frame(
    nombre="Cliente_Marina",
    slots={
        "nombre": "Marina",
        "apellido": "Rodríguez",
        "edad": 35,
        "direccion": "Heredia, Heredia",
        # Atributos CRM
        "numero_cuenta": "CTA-004",
        "arquetipo": "emprendedor_pequeño_empresario",
        "rfm-monetary": "medio",
        "frequency": "baja",
        "recency": "antiguo",
        "customer-journey": "riesgo_de_abandono",
        "afinidad_producto": "credito_personal",
        "canal_contacto": "telefono",
        "saldo": 75000.00
    },
    padre=persona,
    arquetipo="emprendedor_pequeño_empresario"
)

# Cliente 5: Transaccional Básico
cliente_transaccional = Frame(
    nombre="Cliente_Juan",
    slots={
        "nombre": "Juan",
        "apellido": "Pérez",
        "edad": 22,
        "direccion": "San Pedro, San José",
        # Atributos CRM
        "numero_cuenta": "CTA-005",
        "arquetipo": "cliente_transaccional",
        "rfm-monetary": "bajo",
        "frequency": "baja",
        "recency": "inactivo",
        "customer-journey": "reactivacion",
        "afinidad_producto": "cuenta_corriente",
        "canal_contacto": "sms",
        "saldo": 5000.00
    },
    padre=persona,
    arquetipo="cliente_transaccional"
)

clientes_frames = [
    cliente_patrimonial,
    cliente_digital,
    cliente_familia,
    cliente_emprendedor,
    cliente_transaccional
]


## Aplicación de Reglas CRM usando Frames

Demostramos cómo convertir los atributos de un Frame cliente en hechos para el motor de inferencia
y luego aplicar las reglas del sistema CRM.

In [37]:
# ============================================================================
# APLICAR REGLAS CRM A FRAMES DE CLIENTES
# ============================================================================

def aplicar_reglas_a_frame(cliente_frame: Frame, sistema_crm: SistemaCRM) -> Dict[str, Any]:
    """
    Aplica las reglas del sistema CRM a un cliente representado como Frame.

    Args:
        cliente_frame: Frame del cliente
        sistema_crm: Sistema CRM con las reglas cargadas

    Returns:
        Diccionario con información del cliente y reglas aplicables
    """

    # Convertir slots del frame a hechos CRM
    hechos = cliente_frame.get_hechos_crm()

    # También agregar informacion derivada del arquetipo
    hechos.append(f"arquetipo_{cliente_frame.get_slot('arquetipo')}")

    # Agregar hechos basados en el customer journey
    journey = cliente_frame.get_slot('customer-journey')
    if journey:
        hechos.append(f"customer-journey-{journey}")

    # Cargar hechos en el sistema CRM
    sistema_crm.limpiar_hechos()
    sistema_crm.agregar_hechos(hechos)

    # Ejecutar inferencia
    reglas_aplicables = sistema_crm.inferir()

    return {
        'cliente': cliente_frame.nombre,
        'slots': cliente_frame.get_all_slots(),
        'hechos': hechos,
        'reglas_aplicables': reglas_aplicables
    }


# Inicializar sistema CRM con las 45 reglas
sistema_crm = SistemaCRM()

print("\n" + "="*80)
print("APLICACIÓN DE REGLAS CRM A CLIENTES REPRESENTADOS COMO FRAMES")
print("="*80)

# Aplicar reglas a cada cliente
resultados = []
for cliente in clientes_frames:
    resultado = aplicar_reglas_a_frame(cliente, sistema_crm)
    resultados.append(resultado)

    # Mostrar resumen
    print(f"\n{'─'*80}")
    print(f"CLIENTE: {cliente.nombre}")
    print(f"{'─'*80}")
    print(f"Persona: {cliente.get_slot('nombre')} {cliente.get_slot('apellido')}, {cliente.get_slot('edad')} años")
    print(f"Dirección: {cliente.get_slot('direccion')} (HEREDADO de Persona)")
    print(f"Arqueipo: {cliente.arquetipo}")
    print(f"Cuenta: {cliente.get_slot('numero_cuenta')} | Saldo: ${cliente.get_slot('saldo'):,.2f}")
    print(f"\nHechos identificados: {len(resultado['hechos'])} hechos")
    print(f"Reglas aplicables: {len(resultado['reglas_aplicables'])} regla(s)")

    if resultado['reglas_aplicables']:
        regla_optima = resultado['reglas_aplicables'][0]
        print(f"\nREGLA SELECCIONADA (Máxima Especificidad):")
        print(f"   ID: {regla_optima['id']}")
        print(f"   Condiciones: {len(regla_optima['si'])}")
        print(f"   Acción: {regla_optima['entonces']}")

print("\n" + "="*80)
print("APLICACIÓN DE REGLAS COMPLETADA")
print("="*80)



APLICACIÓN DE REGLAS CRM A CLIENTES REPRESENTADOS COMO FRAMES

────────────────────────────────────────────────────────────────────────────────
CLIENTE: Cliente_Roberto
────────────────────────────────────────────────────────────────────────────────
Persona: Roberto Martínez, 55 años
Dirección: Escalante, San José (HEREDADO de Persona)
Arqueipo: cliente_patrimonial
Cuenta: CTA-001 | Saldo: $500,000.00

Hechos identificados: 13 hechos
Reglas aplicables: 0 regla(s)

────────────────────────────────────────────────────────────────────────────────
CLIENTE: Cliente_Andrea
────────────────────────────────────────────────────────────────────────────────
Persona: Andrea González, 28 años
Dirección: Barrio Escalante, San José (HEREDADO de Persona)
Arqueipo: profesional_joven_digital
Cuenta: CTA-002 | Saldo: $15,000.00

Hechos identificados: 13 hechos
Reglas aplicables: 0 regla(s)

────────────────────────────────────────────────────────────────────────────────
CLIENTE: Cliente_familia_Lopez
──

## Demostración mediante reglas de producción

In [38]:
# ========== EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición) ==========
print("\n" + "="*65)
print("EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm-monetary-bajo",
    "frequency_baja",
    "recency_muy_reciente",
    "arquetipo_cliente_transaccional",
    "customer-journey-adquisicion",
    "afinidad_cuenta_corriente",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Cliente Transaccional #001")


EJEMPLO 1: Cliente Transaccional - Bajo Valor (Adquisición)

+==============================================================+
| ANALISIS CRM: Cliente Transaccional #001                         |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm-monetary-bajo
   - frequency_baja
   - recency_muy_reciente
   - arquetipo_cliente_transaccional
   - customer-journey-adquisicion
   - afinidad_cuenta_corriente
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

*****************************************************************
[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)
*****************************************************************
Regla ID: R28
Especificidad: ALTA (6-7) (7 condiciones)

[ACCION] ACCION RECOMENDADA:
  >> Campaña automática: Cuenta corriente con beneficios básicos
**************************************************************

[{'id': 'R28',
  'si': ['rfm-monetary-bajo',
   'frequency_baja',
   'recency_muy_reciente',
   'arquetipo_cliente_transaccional',
   'customer-journey-adquisicion',
   'afinidad_cuenta_corriente',
   'canal_aplicacion_movil'],
  'entonces': 'Campaña automática: Cuenta corriente con beneficios básicos'}]

El sistema ahora opera en dos modos según la especificidad de la regla:

### MODO ALTA ESPECIFICIDAD (6-7 variables)
Cuando la mejor regla tiene 6-7 variables:
- [OK] Muestra recomendacion completa
- [OK] Muestra trazabilidad (audit trail) de todas las reglas

### MODO MENOR ESPECIFICIDAD (5 o 4 variables)
Cuando la mejor regla tiene 5 o 4 variables:
- [NOTA] Solo muestra trazabilidad (sin recomendacion directa)
- [INFO] Permite auditoria de decisiones

In [39]:
# ========== DEMOSTRACIÓN: MODO ALTA ESPECIFICIDAD (6-7 variables) ==========
print("\n" + "="*70)
print("DEMOSTRACIÓN 1: MODO ALTA ESPECIFICIDAD (6-7 variables)")
print("Resultado: Recomendación COMPLETA + Trazabilidad")
print("="*70 + "\n")

sistema.limpiar_hechos()
# Agregar 6-7 variables para asegurar especificidad ALTA
sistema.agregar_hechos([
    "rfm-monetary-bajo",
    "frequency_baja",
    "recency_muy_reciente",
    "arquetipo_cliente_transaccional",
    "customer-journey-adquisicion",
    "afinidad_cuenta_corriente",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Cliente Transaccional - ALTA ESPECIFICIDAD (7 variables)")



DEMOSTRACIÓN 1: MODO ALTA ESPECIFICIDAD (6-7 variables)
Resultado: Recomendación COMPLETA + Trazabilidad

+==============================================================+
| ANALISIS CRM: Cliente Transaccional - ALTA ESPECIFICIDAD (7 variables) |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm-monetary-bajo
   - frequency_baja
   - recency_muy_reciente
   - arquetipo_cliente_transaccional
   - customer-journey-adquisicion
   - afinidad_cuenta_corriente
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

*****************************************************************
[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)
*****************************************************************
Regla ID: R28
Especificidad: ALTA (6-7) (7 condiciones)

[ACCION] ACCION RECOMENDADA:
  >> Campaña automática: Cuenta corriente con beneficios básicos
***********

[{'id': 'R28',
  'si': ['rfm-monetary-bajo',
   'frequency_baja',
   'recency_muy_reciente',
   'arquetipo_cliente_transaccional',
   'customer-journey-adquisicion',
   'afinidad_cuenta_corriente',
   'canal_aplicacion_movil'],
  'entonces': 'Campaña automática: Cuenta corriente con beneficios básicos'}]

In [40]:
# ========== DEMOSTRACIÓN: MODO MENOR ESPECIFICIDAD (4 variables) ==========
print("\n" + "="*70)
print("DEMOSTRACIÓN 2: MODO MENOR ESPECIFICIDAD (4 variables)")
print("Resultado: SOLO Trazabilidad (SIN recomendación directa)")
print("="*70 + "\n")

sistema.limpiar_hechos()
# Agregar solo 4 variables (sin canal) para especificidad BAJA
sistema.agregar_hechos([
    "rfm-monetary-alto",
    "arquetipo_cliente_patrimonial",
    "customer-journey-crecimiento",
    "afinidad_fondos_inversion"
])
sistema.mostrar_resultado("Patrimonial Alto - BAJA ESPECIFICIDAD (4 variables, sin canal)")



DEMOSTRACIÓN 2: MODO MENOR ESPECIFICIDAD (4 variables)
Resultado: SOLO Trazabilidad (SIN recomendación directa)

+==============================================================+
| ANALISIS CRM: Patrimonial Alto - BAJA ESPECIFICIDAD (4 variables, sin canal) |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (4 condiciones):
   - rfm-monetary-alto
   - arquetipo_cliente_patrimonial
   - customer-journey-crecimiento
   - afinidad_fondos_inversion

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

[NOTA] NOTA: Especificidad BAJA (4) (4 variables)
Se muestran las reglas usadas por trazabilidad (sin recomendacion).

─────────────────────────────────────────────────────────────────
[TRAZABILIDAD] TRAZABILIDAD: Reglas Aplicables (1 total):
─────────────────────────────────────────────────────────────────

1. R66 [OPTIMA]
   Especificidad: BAJA (4) (4 condiciones)
   Condiciones: rfm-monetary-alto, arquetipo_clien

[{'id': 'R66',
  'si': ['rfm-monetary-alto',
   'arquetipo_cliente_patrimonial',
   'customer-journey-crecimiento',
   'afinidad_fondos_inversion'],
  'entonces': 'Asesoría de inversión personalizada por valor alto'}]

In [41]:
# ========== DEMOSTRACIÓN: MODO MEDIANA ESPECIFICIDAD (5 variables) ==========
print("\n" + "="*70)
print("DEMOSTRACIÓN 3: MODO MEDIANA ESPECIFICIDAD (5 variables)")
print("Resultado: SOLO Trazabilidad (SIN recomendación directa)")
print("="*70 + "\n")

sistema.limpiar_hechos()
# Agregar 5 variables para especificidad MEDIA
sistema.agregar_hechos([
    "rfm-monetary-medio",
    "arquetipo_profesional_joven_digital",
    "customer-journey-crecimiento",
    "afinidad_fondos_inversion",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Digital Medio - MEDIA ESPECIFICIDAD (5 variables)")



DEMOSTRACIÓN 3: MODO MEDIANA ESPECIFICIDAD (5 variables)
Resultado: SOLO Trazabilidad (SIN recomendación directa)

+==============================================================+
| ANALISIS CRM: Digital Medio - MEDIA ESPECIFICIDAD (5 variables)  |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (5 condiciones):
   - rfm-monetary-medio
   - arquetipo_profesional_joven_digital
   - customer-journey-crecimiento
   - afinidad_fondos_inversion
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

[NOTA] NOTA: Especificidad BAJA (4) (4 variables)
Se muestran las reglas usadas por trazabilidad (sin recomendacion).

─────────────────────────────────────────────────────────────────
[TRAZABILIDAD] TRAZABILIDAD: Reglas Aplicables (1 total):
─────────────────────────────────────────────────────────────────

1. R73 [OPTIMA]
   Especificidad: BAJA (4) (4 condiciones)
   Condiciones: rfm-monet

[{'id': 'R73',
  'si': ['rfm-monetary-medio',
   'arquetipo_profesional_joven_digital',
   'customer-journey-crecimiento',
   'afinidad_fondos_inversion'],
  'entonces': 'Plataforma inversiones progresiva'}]

In [42]:
print("\n" + "="*85)
print("RESUMEN: LOGICA DE DECISION DEL SISTEMA CRM (Modos de Operacion)")
print("="*85)

resumen_datos = {
    'Especificidad': ['ALTA (6-7 variables)', 'MEDIA (5 variables)', 'BAJA (4 variables)'],
    'Rango de Reglas': ['R1-R45*', 'R46-R65', 'R66-R95'],
    'Mostrar Recomendacion': ['[SI]', '[NO]', '[NO]'],
    'Mostrar Trazabilidad': ['[SI] (Completa', '[SI] (Completa)', '[SI] (Completa)'],
    'Incluye': ['frequency, recency, canal', 'sin frequency/recency, con canal', 'sin canal, solo 4 vars'],
    'Caso de Uso': ['Recomendacion directa al cliente', 'Auditoria + derivacion manual', 'Auditoria + fallback']
}

df_resumen = pd.DataFrame(resumen_datos)
print("\n")
print(df_resumen.to_string(index=False))

print("\n" + "="*85)
print("EXPLICACION DE LA LOGICA:")
print("="*85)
print("""
[ALTA] ESPECIFICIDAD ALTA (6-7 variables):
   - Suficiente informacion para hacer una RECOMENDACION COMPLETA
   - Resultado: Accion recomendada clara + Trazabilidad de todas las reglas
   - Uso: Decisiones automaticas o directas al cliente

[MEDIA] ESPECIFICIDAD MEDIA (5 variables):
   - Informacion incompleta (faltan frequency o recency)
   - Resultado: SOLO Trazabilidad, SIN recomendacion directa
   - Uso: Auditoria interna, revision manual, derivacion a experto

[BAJA] ESPECIFICIDAD BAJA (4 variables):
   - Informacion minima, sin canal definido
   - Resultado: SOLO Trazabilidad como fallback
   - Uso: Ultimo recurso, casos especiales, escalacion requerida

VENTAJAS DE ESTE ESQUEMA:
[OK] Transparencia: Usuario ve exactamente que reglas se aplicaron
[OK] Confiabilidad: Solo recomienda cuando hay suficiente base
[OK] Trazabilidad: Auditoria completa de todas las decisiones
[OK] Escalabilidad: Permite derivacion automatica segun especificidad
""")
print("="*85 + "\n")


RESUMEN: LOGICA DE DECISION DEL SISTEMA CRM (Modos de Operacion)


       Especificidad Rango de Reglas Mostrar Recomendacion Mostrar Trazabilidad                          Incluye                      Caso de Uso
ALTA (6-7 variables)         R1-R45*                  [SI]       [SI] (Completa        frequency, recency, canal Recomendacion directa al cliente
 MEDIA (5 variables)         R46-R65                  [NO]      [SI] (Completa) sin frequency/recency, con canal    Auditoria + derivacion manual
  BAJA (4 variables)         R66-R95                  [NO]      [SI] (Completa)           sin canal, solo 4 vars             Auditoria + fallback

EXPLICACION DE LA LOGICA:

[ALTA] ESPECIFICIDAD ALTA (6-7 variables):
   - Suficiente informacion para hacer una RECOMENDACION COMPLETA
   - Resultado: Accion recomendada clara + Trazabilidad de todas las reglas
   - Uso: Decisiones automaticas o directas al cliente

[MEDIA] ESPECIFICIDAD MEDIA (5 variables):
   - Informacion incompleta (faltan 

In [43]:
# ========== EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento) ==========
print("\n" + "="*65)
print("EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm-monetary-alto",
    "frequency_alta",
    "arquetipo_cliente_patrimonial",
    "customer-journey-crecimiento",
    "afinidad_fondos_inversion",
    "canal_asesor_personal"
])
sistema.mostrar_resultado("Cliente Patrimonial #002")


EJEMPLO 2: Cliente Patrimonial - Alto Valor (Crecimiento)

+==============================================================+
| ANALISIS CRM: Cliente Patrimonial #002                           |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (6 condiciones):
   - rfm-monetary-alto
   - frequency_alta
   - arquetipo_cliente_patrimonial
   - customer-journey-crecimiento
   - afinidad_fondos_inversion
   - canal_asesor_personal

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 3 regla(s):

*****************************************************************
[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)
*****************************************************************
Regla ID: R9
Especificidad: ALTA (6-7) (6 condiciones)

[ACCION] ACCION RECOMENDADA:
  >> Ofrecer asesoría financiera personalizada
*****************************************************************

─────────────────────────────────────────────

[{'id': 'R9',
  'si': ['rfm-monetary-alto',
   'frequency_alta',
   'arquetipo_cliente_patrimonial',
   'customer-journey-crecimiento',
   'afinidad_fondos_inversion',
   'canal_asesor_personal'],
  'entonces': 'Ofrecer asesoría financiera personalizada'},
 {'id': 'R1',
  'si': ['rfm-monetary-alto',
   'arquetipo_cliente_patrimonial',
   'customer-journey-crecimiento',
   'afinidad_fondos_inversion',
   'canal_asesor_personal'],
  'entonces': 'Ofrecer productos de inversión premium'},
 {'id': 'R66',
  'si': ['rfm-monetary-alto',
   'arquetipo_cliente_patrimonial',
   'customer-journey-crecimiento',
   'afinidad_fondos_inversion'],
  'entonces': 'Asesoría de inversión personalizada por valor alto'}]

In [44]:
# ========== EJEMPLO 3: Cliente Digital - Riesgo de Abandono ==========
print("\n" + "="*65)
print("EJEMPLO 3: Cliente Digital - Riesgo de Abandono (Retención)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm-monetary-bajo",
    "frequency_baja",
    "recency_antiguo",
    "arquetipo_profesional_joven_digital",
    "customer-journey-riesgo_de_abandono",
    "afinidad_tarjeta_credito",
    "canal_aplicacion_movil"
])
sistema.mostrar_resultado("Cliente Digital #003 - EN RIESGO")


EJEMPLO 3: Cliente Digital - Riesgo de Abandono (Retención)

+==============================================================+
| ANALISIS CRM: Cliente Digital #003 - EN RIESGO                   |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm-monetary-bajo
   - frequency_baja
   - recency_antiguo
   - arquetipo_profesional_joven_digital
   - customer-journey-riesgo_de_abandono
   - afinidad_tarjeta_credito
   - canal_aplicacion_movil

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

*****************************************************************
[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)
*****************************************************************
Regla ID: R29
Especificidad: ALTA (6-7) (7 condiciones)

[ACCION] ACCION RECOMENDADA:
  >> Campaña automática: Oferta de tarjeta crédito sin anualidad
*********************************************************

[{'id': 'R29',
  'si': ['rfm-monetary-bajo',
   'frequency_baja',
   'recency_antiguo',
   'arquetipo_profesional_joven_digital',
   'customer-journey-riesgo_de_abandono',
   'afinidad_tarjeta_credito',
   'canal_aplicacion_movil'],
  'entonces': 'Campaña automática: Oferta de tarjeta crédito sin anualidad'}]

In [45]:
# ========== EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento) ==========
print("\n" + "="*65)
print("EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento)")
print("="*65 + "\n")

sistema.limpiar_hechos()
sistema.agregar_hechos([
    "rfm-monetary-medio",
    "frequency_baja",
    "recency_antiguo",
    "arquetipo_emprendedor_pequeño_empresario",
    "customer-journey-crecimiento",
    "afinidad_credito_personal",
    "canal_telefono"
])
sistema.mostrar_resultado("Emprendedor #004")



EJEMPLO 4: Emprendedor - Valor Medio (Crecimiento)

+==============================================================+
| ANALISIS CRM: Emprendedor #004                                   |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (7 condiciones):
   - rfm-monetary-medio
   - frequency_baja
   - recency_antiguo
   - arquetipo_emprendedor_pequeño_empresario
   - customer-journey-crecimiento
   - afinidad_credito_personal
   - canal_telefono

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 2 regla(s):

*****************************************************************
[OK] RECOMENDACION: REGLA SELECCIONADA (Maxima Especificidad)
*****************************************************************
Regla ID: R19
Especificidad: ALTA (6-7) (7 condiciones)

[ACCION] ACCION RECOMENDADA:
  >> Ofrecer líneas de crédito comercial
*****************************************************************

───────────────────────────────

[{'id': 'R19',
  'si': ['rfm-monetary-medio',
   'frequency_baja',
   'recency_antiguo',
   'arquetipo_emprendedor_pequeño_empresario',
   'customer-journey-crecimiento',
   'afinidad_credito_personal',
   'canal_telefono'],
  'entonces': 'Ofrecer líneas de crédito comercial'},
 {'id': 'R6',
  'si': ['rfm-monetary-medio',
   'arquetipo_emprendedor_pequeño_empresario',
   'customer-journey-crecimiento',
   'afinidad_credito_personal',
   'canal_telefono'],
  'entonces': 'Consultoría de crédito comercial personalizada'}]

## Ejemplos con Nuevas Reglas (R46-R65): 5 Variables Mínimas

Demostramos cómo funcionan las nuevas reglas simplificadas que utilizan solo las 5 variables mínimas esenciales.

## Nuevas Reglas (R66-R95): 4 Variables Mínimas (SIN CANAL)

Estas 30 reglas nuevas utilizan solo 4 variables esenciales:
- **rfm-monetary**: Valor monetario del cliente (alto, medio, bajo)
- **arquetipo**: Tipo de cliente (5 arquetipos)
- **customer-journey**: Etapa del ciclo de vida (6 etapas)
- **afinidad**: Producto/Servicio (5 afinidades)

Sin incluir el canal de contacto, estas reglas tienen especificidad **BAJA** pero amplían significativamente la cobertura del sistema de 65 a 95 reglas.

In [46]:
# ========== EJEMPLO 1: R66 - Patrimonial + Fondos + Crecimiento (SIN CANAL) ==========
print("\n" + "=" * 70)
print("EJEMPLO 1: R66 - Patrimonial + Inversión + Crecimiento (4 variables)")
print("=" * 70 + "\n")

sistema_extendido.limpiar_hechos()
sistema_extendido.agregar_hechos([
    "rfm-monetary-alto",
    "arquetipo_cliente_patrimonial",
    "customer-journey-crecimiento",
    "afinidad_fondos_inversion"
])
sistema_extendido.mostrar_resultado("Patrimonial Alto - R66 (Sin Canal)")

# ========== EJEMPLO 2: R73 - Digital + Fondos + Crecimiento (SIN CANAL) ==========
print("\n" + "=" * 70)
print("EJEMPLO 2: R73 - Digital + Inversión + Crecimiento (4 variables)")
print("=" * 70 + "\n")

sistema_extendido.limpiar_hechos()
sistema_extendido.agregar_hechos([
    "rfm-monetary-medio",
    "arquetipo_profesional_joven_digital",
    "customer-journey-crecimiento",
    "afinidad_fondos_inversion"
])
sistema_extendido.mostrar_resultado("Digital Medio - R73 (Sin Canal)")

# ========== EJEMPLO 3: R80 - Familia + Seguros + Madurez (SIN CANAL) ==========
print("\n" + "=" * 70)
print("EJEMPLO 3: R80 - Familia + Seguros + Madurez (4 variables)")
print("=" * 70 + "\n")

sistema_extendido.limpiar_hechos()
sistema_extendido.agregar_hechos([
    "rfm-monetary-bajo",
    "arquetipo_familia_en_expansion",
    "customer-journey-madurez",
    "afinidad_seguros"
])
sistema_extendido.mostrar_resultado("Familia Bajo - R80 (Sin Canal)")

# ========== EJEMPLO 4: R91 - Familia + Seguros + Crecimiento (SIN CANAL) ==========
print("\n" + "=" * 70)
print("EJEMPLO 4: R91 - Familia + Seguros + Crecimiento (4 variables)")
print("=" * 70 + "\n")

sistema_extendido.limpiar_hechos()
sistema_extendido.agregar_hechos([
    "rfm-monetary-medio",
    "arquetipo_familia_en_expansion",
    "customer-journey-crecimiento",
    "afinidad_seguros"
])
sistema_extendido.mostrar_resultado("Familia Medio - R91 (Sin Canal)")

# ========== EJEMPLO 5: R87 - Transaccional + Fondos + Madurez (SIN CANAL) ==========
print("\n" + "=" * 70)
print("EJEMPLO 5: R87 - Transaccional + Inversión + Madurez (4 variables)")
print("=" * 70 + "\n")

sistema_extendido.limpiar_hechos()
sistema_extendido.agregar_hechos([
    "rfm-monetary-alto",
    "arquetipo_cliente_transaccional",
    "customer-journey-madurez",
    "afinidad_fondos_inversion"
])
sistema_extendido.mostrar_resultado("Transaccional Alto - R87 (Sin Canal)")



EJEMPLO 1: R66 - Patrimonial + Inversión + Crecimiento (4 variables)

+==============================================================+
| ANALISIS CRM: Patrimonial Alto - R66 (Sin Canal)                 |
+==============================================================+

[HECHOS] HECHOS IDENTIFICADOS (4 condiciones):
   - rfm-monetary-alto
   - arquetipo_cliente_patrimonial
   - customer-journey-crecimiento
   - afinidad_fondos_inversion

[BUSCANDO] BUSCANDO REGLAS APLICABLES...

[ENCONTRADAS] Se encontraron 1 regla(s):

[NOTA] NOTA: Especificidad BAJA (4) (4 variables)
Se muestran las reglas usadas por trazabilidad (sin recomendacion).

─────────────────────────────────────────────────────────────────
[TRAZABILIDAD] TRAZABILIDAD: Reglas Aplicables (1 total):
─────────────────────────────────────────────────────────────────

1. R66 [OPTIMA]
   Especificidad: BAJA (4) (4 condiciones)
   Condiciones: rfm-monetary-alto, arquetipo_cliente_patrimonial, customer-journey-crecimiento...
   Acci

[{'id': 'R87',
  'si': ['rfm-monetary-alto',
   'arquetipo_cliente_transaccional',
   'customer-journey-madurez',
   'afinidad_fondos_inversion'],
  'entonces': 'Portafolio inversión para cliente activo'}]

## Resumen de Todas las 95 Reglas del Sistema CRM

In [47]:
# ============================================================================
# ANÁLISIS COMPLETO: 95 Reglas (R1-R95)
# ============================================================================

print("\n" + "=" * 100)
print("ANÁLISIS COMPLETO DEL SISTEMA CRM EXTENDIDO: 95 REGLAS (R1-R95)")
print("=" * 100 + "\n")

# Calcular distribución por número de condiciones
distribucion_final = {}
for regla in TODAS_LAS_REGLAS:
    num_cond = len(regla['si'])
    if num_cond not in distribucion_final:
        distribucion_final[num_cond] = []
    distribucion_final[num_cond].append(regla['id'])

# Tabla de distribución
print("DISTRIBUCIÓN POR ESPECIFICIDAD:")
print("-" * 100)
tabla_distribucion = []
for num_cond in sorted(distribucion_final.keys()):
    cantidad = len(distribucion_final[num_cond])
    porcentaje = (cantidad / len(TODAS_LAS_REGLAS) * 100)
    especificidad = "ALTA" if num_cond >= 6 else "MEDIA" if num_cond == 5 else "BAJA"
    reglas = ', '.join(distribucion_final[num_cond][:10])
    if len(distribucion_final[num_cond]) > 10:
        reglas += f", ... ({cantidad - 10} más)"
    
    tabla_distribucion.append({
        'Variables': num_cond,
        'Cantidad': cantidad,
        'Porcentaje': f'{porcentaje:.1f}%',
        'Especificidad': especificidad,
        'Ejemplos': reglas
    })

df_dist = pd.DataFrame(tabla_distribucion)
print(df_dist.to_string(index=False))

print("\n" + "-" * 100)
print(f"\nTOTAL DE REGLAS: {len(TODAS_LAS_REGLAS)}")
print(f"RANGO DE VARIABLES: {min(distribucion_final.keys())} a {max(distribucion_final.keys())} por regla")
promedio_vars = sum(len(r['si']) for r in TODAS_LAS_REGLAS) / len(TODAS_LAS_REGLAS)
print(f"PROMEDIO DE VARIABLES POR REGLA: {promedio_vars:.2f}")

print("\n" + "=" * 100)
print("DESGLOSE POR NIVEL DE ESPECIFICIDAD:")
print("=" * 100)

# Reglas de especificidad ALTA (6-7 variables)
reglas_alta = [r for r in TODAS_LAS_REGLAS if len(r['si']) >= 6]
print(f"\n✓ ESPECIFICIDAD ALTA (6-7 variables): {len(reglas_alta)} reglas")
print(f"  Rango: {[r['id'] for r in reglas_alta][:5]} ... {[r['id'] for r in reglas_alta][-5:]}")
print(f"  Uso: Mayor prioridad en resolución de conflictos")
print(f"  Características: Incluyen frequency, recency y/o canal")

# Reglas de especificidad MEDIA (5 variables)
reglas_media = [r for r in TODAS_LAS_REGLAS if len(r['si']) == 5]
print(f"\n✓ ESPECIFICIDAD MEDIA (5 variables): {len(reglas_media)} reglas")
print(f"  Rango: R1-R65 (primeras 45) + R46-R65 (nuevas 20)")
print(f"  Uso: Aplicadas cuando no hay reglas de especificidad alta")
print(f"  Variables base: rfm-monetary, arquetipo, customer-journey, afinidad")

# Reglas de especificidad BAJA (4 variables)
reglas_baja = [r for r in TODAS_LAS_REGLAS if len(r['si']) == 4]
print(f"\n✓ ESPECIFICIDAD BAJA (4 variables): {len(reglas_baja)} reglas")
print(f"  Rango: R66-R95 (nuevas 30)")
print(f"  Uso: Fallback (aplicadas como último recurso)")
print(f"  Características: SIN canal (mayor cobertura)")

print("\n" + "=" * 100)
print("ESTADÍSTICAS POR DIMENSIÓN:")
print("=" * 100)

# RFM Monetary
print("\n1. POR RFM MONETARY:")
for rfm in ['alto', 'medio', 'bajo']:
    count = len([r for r in TODAS_LAS_REGLAS if any(f'rfm-monetary-{rfm}' in c for c in r['si'])])
    print(f"   {rfm.upper():6} → {count:2d} reglas")

# Arquetipos
print("\n2. POR ARQUETIPO:")
arquetipos_count = {}
for arq in ['cliente_patrimonial', 'profesional_joven_digital', 'familia_en_expansion', 'cliente_transaccional', 'emprendedor_pequeño_empresario']:
    count = len([r for r in TODAS_LAS_REGLAS if any(f'arquetipo_{arq}' in c for c in r['si'])])
    arquetipos_count[arq] = count
    print(f"   {arq:35} → {count:2d} reglas")

# Customer Journey
print("\n3. POR CUSTOMER JOURNEY:")
for j in ['adquisicion', 'activacion', 'crecimiento', 'madurez', 'reactivacion', 'riesgo_de_abandono']:
    count = len([r for r in TODAS_LAS_REGLAS if any(f'customer-journey-{j}' in c for c in r['si'])])
    print(f"   {j:20} → {count:2d} reglas")

# Afinidades
print("\n4. POR AFINIDAD:")
for af in ['fondos_inversion', 'tarjeta_credito', 'seguros', 'cuenta_corriente', 'credito_personal']:
    count = len([r for r in TODAS_LAS_REGLAS if any(f'afinidad_{af}' in c for c in r['si'])])
    print(f"   {af:18} → {count:2d} reglas")

# Canales (solo en reglas con 5+ variables)
print("\n5. POR CANAL (solo en reglas de 5+ variables):")
reglas_con_canal = [r for r in TODAS_LAS_REGLAS if len(r['si']) >= 5]
for canal in ['asesor_personal', 'aplicacion_movil', 'email', 'sms', 'telefono', 'sucursal']:
    count = len([r for r in reglas_con_canal if any(f'canal_{canal}' in c for c in r['si'])])
    if count > 0:
        print(f"   {canal:18} → {count:2d} reglas")

print("\n" + "=" * 100)
print("RESUMEN EJECUTIVO:")
print("=" * 100)
print(f"""
El sistema CRM ahora cuenta con 95 reglas de decisión distribuidas en tres niveles de especificidad:

📊 NIVEL ALTO (25 reglas con 6-7 variables):
   • Mayor precisión y contexto
   • Incluyen factores adicionales: frequency, recency, canal
   • Prioritarias en resolución de conflictos

📊 NIVEL MEDIO (40 reglas con 5 variables):
   • Equilibrio entre especificidad y cobertura
   • Base: rfm-monetary, arquetipo, customer-journey, afinidad, canal
   • R1-R65: Reglas existentes + nuevas

📊 NIVEL BAJO (30 reglas con 4 variables):
   • Mayor cobertura (SIN canal)
   • Caso fallback para escenarios no cubiertos
   • R66-R95: Nuevas reglas

✓ COBERTURA TOTAL:
   • 5 arquetipos de cliente
   • 3 niveles RFM monetary
   • 6 etapas de customer journey
   • 5 afinidades de producto
   • 6 canales de contacto (en reglas nivel alto/medio)
""")
print("=" * 100)



ANÁLISIS COMPLETO DEL SISTEMA CRM EXTENDIDO: 95 REGLAS (R1-R95)

DISTRIBUCIÓN POR ESPECIFICIDAD:
----------------------------------------------------------------------------------------------------
 Variables  Cantidad Porcentaje Especificidad                                                       Ejemplos
         4        30      31.6%          BAJA R66, R67, R68, R69, R70, R71, R72, R73, R74, R75, ... (20 más)
         5        28      29.5%         MEDIA         R1, R2, R3, R4, R5, R6, R7, R8, R46, R47, ... (18 más)
         6         5       5.3%          ALTA                                         R9, R11, R12, R13, R14
         7        32      33.7%          ALTA R10, R15, R16, R17, R18, R19, R20, R21, R22, R23, ... (22 más)

----------------------------------------------------------------------------------------------------

TOTAL DE REGLAS: 95
RANGO DE VARIABLES: 4 a 7 por regla
PROMEDIO DE VARIABLES POR REGLA: 5.41

DESGLOSE POR NIVEL DE ESPECIFICIDAD:

✓ ESPECIFICIDAD ALTA

In [48]:
# ============================================================================
# TABLA: Nuevas Reglas (R46-R95)
# ============================================================================

import pandas as pd

print("\n" + "=" * 130)
print("LISTADO COMPLETO DE NUEVAS REGLAS (R46-R95): 5 Y 4 VARIABLES MÍNIMAS")
print("=" * 130 + "\n")

# Separar reglas por nivel de especificidad
reglas_5vars = [r for r in TODAS_LAS_REGLAS if r['id'].startswith('R') and (46 <= int(r['id'][1:]) <= 65) and len(r['si']) == 5]
reglas_4vars = [r for r in TODAS_LAS_REGLAS if r['id'].startswith('R') and (66 <= int(r['id'][1:]) <= 95) and len(r['si']) == 4]

# ========== TABLA R46-R65 (5 VARIABLES) ==========
print("📊 NUEVAS REGLAS R46-R65: 5 VARIABLES MÍNIMAS (Especificidad MEDIA)")
print("-" * 130)

tabla_5vars = []
for regla in reglas_5vars:
    rfm = next((r.split('_')[-1] for r in regla['si'] if r.startswith('rfm-monetary')), '')
    arquetipo = next((r.split('_', 2)[-1] for r in regla['si'] if r.startswith('arquetipo')), '')
    journey = next((r.split('_', 1)[-1] for r in regla['si'] if r.startswith('customer-journey')), '')
    afinidad = next((r.split('_', 1)[-1] for r in regla['si'] if r.startswith('afinidad')), '')
    canal = next((r.split('_', 1)[-1] for r in regla['si'] if r.startswith('canal')), '')
    action = regla['entonces'][:60] + '...' if len(regla['entonces']) > 60 else regla['entonces']
    
    tabla_5vars.append({
        'ID': regla['id'],
        'RFM': rfm,
        'Arquetipo': arquetipo[:20],
        'Journey': journey,
        'Afinidad': afinidad,
        'Canal': canal,
        'Acción': action
    })

df_5vars = pd.DataFrame(tabla_5vars)
print(df_5vars.to_string(index=False))
print(f"\nTotal R46-R65: {len(reglas_5vars)} reglas con 5 variables")

# ========== TABLA R66-R95 (4 VARIABLES) ==========
print("\n" + "=" * 130)
print("📊 NUEVAS REGLAS R66-R95: 4 VARIABLES MÍNIMAS SIN CANAL (Especificidad BAJA)")
print("-" * 130)

tabla_4vars = []
for regla in reglas_4vars:
    rfm = next((r.split('_')[-1] for r in regla['si'] if r.startswith('rfm-monetary')), '')
    arquetipo = next((r.split('_', 2)[-1] for r in regla['si'] if r.startswith('arquetipo')), '')
    journey = next((r.split('_', 1)[-1] for r in regla['si'] if r.startswith('customer-journey')), '')
    afinidad = next((r.split('_', 1)[-1] for r in regla['si'] if r.startswith('afinidad')), '')
    action = regla['entonces'][:55] + '...' if len(regla['entonces']) > 55 else regla['entonces']
    
    tabla_4vars.append({
        'ID': regla['id'],
        'RFM': rfm,
        'Arquetipo': arquetipo[:20],
        'Journey': journey,
        'Afinidad': afinidad,
        'Acción': action
    })

# Mostrar en chunks de 10 reglas
for i in range(0, len(tabla_4vars), 10):
    chunk_df = pd.DataFrame(tabla_4vars[i:i+10])
    print(f"\n[Reglas {tabla_4vars[i]['ID']} - {tabla_4vars[min(i+9, len(tabla_4vars)-1)]['ID']}]")
    print(chunk_df.to_string(index=False))

print(f"\n\nTotal R66-R95: {len(reglas_4vars)} reglas con 4 variables")

print("\n" + "=" * 130)
print("RESUMEN DE NUEVAS REGLAS:")
print("=" * 130)
print(f"""
• R46-R65: {len(reglas_5vars)} reglas con 5 variables (rfm-monetary, arquetipo, customer-journey, afinidad, canal)
• R66-R95: {len(reglas_4vars)} reglas con 4 variables (rfm-monetary, arquetipo, customer-journey, afinidad) [SIN CANAL]

Total nuevas reglas agregadas: {len(reglas_5vars) + len(reglas_4vars)} reglas
Sistema total: 95 reglas (R1-R95)
""")

print("ESTADÍSTICAS POR CATEGORÍA (R46-R95):\n")

# RFM
print("Por RFM Monetary (R46-R95):")
nuevas_reglas = reglas_5vars + reglas_4vars
for rfm in ['alto', 'medio', 'bajo']:
    count = len([r for r in nuevas_reglas if any(f'rfm-monetary-{rfm}' in c for c in r['si'])])
    print(f"  {rfm.upper():6} → {count} reglas")

# Arquetipos
print("\nPor Arquetipo (R46-R95):")
for arq in ['cliente_patrimonial', 'profesional_joven_digital', 'familia_en_expansion', 'cliente_transaccional', 'emprendedor_pequeño_empresario']:
    count = len([r for r in nuevas_reglas if any(f'arquetipo_{arq}' in c for c in r['si'])])
    print(f"  {arq:35} → {count} reglas")

# Journey
print("\nPor Customer Journey (R46-R95):")
for j in ['adquisicion', 'activacion', 'crecimiento', 'madurez', 'reactivacion', 'riesgo_de_abandono']:
    count = len([r for r in nuevas_reglas if any(f'customer-journey-{j}' in c for c in r['si'])])
    if count > 0:
        print(f"  {j:20} → {count} reglas")

print("\n" + "=" * 130)



LISTADO COMPLETO DE NUEVAS REGLAS (R46-R95): 5 Y 4 VARIABLES MÍNIMAS

📊 NUEVAS REGLAS R46-R65: 5 VARIABLES MÍNIMAS (Especificidad MEDIA)
----------------------------------------------------------------------------------------------------------------------------------
 ID                RFM          Arquetipo                       Journey         Afinidad            Canal                                            Acción
R46  rfm-monetary-alto       en_expansion  customer-journey-crecimiento credito_personal         telefono      Crédito familiar con financiamiento flexible
R47  rfm-monetary-bajo      joven_digital  customer-journey-crecimiento  tarjeta_credito            email    Oferta de tarjeta crédito con cashback digital
R48 rfm-monetary-medio        patrimonial   customer-journey-activacion          seguros         sucursal               Consultoría seguros para patrimonio
R49  rfm-monetary-alto pequeño_empresario   customer-journey-activacion fondos_inversion aplicacion_movil  

## Resumen Completo: Nuevas Reglas (R46-R65)

Tabla con todas las nuevas reglas de 5 variables mínimas creadas para el sistema CRM extendido.

In [49]:
# ============================================================================
# TABLA RESUMEN: Nuevas Reglas (R46-R65)
# ============================================================================

import pandas as pd

print("\n" + "=" * 120)
print("LISTADO COMPLETO DE NUEVAS REGLAS (R46-R65): 5 VARIABLES MÍNIMAS")
print("=" * 120 + "\n")

# Crear tabla con las nuevas reglas
tabla_reglas = []
for regla in reglas_nuevas:
    rfm = next((r.split('_')[-1] for r in regla['si'] if r.startswith('rfm-monetary')), '')
    arquetipo = next((r.split('_', 2)[-1] for r in regla['si'] if r.startswith('arquetipo')), '')
    journey = next((r.split('_', 2)[-1] for r in regla['si'] if r.startswith('customer-journey')), '')
    afinidad = next((r.split('_')[-1] for r in regla['si'] if r.startswith('afinidad')), '')
    canal = next((r.split('_')[-1] for r in regla['si'] if r.startswith('canal')), '')
    action = regla['entonces']
    
    tabla_reglas.append({
        'ID': regla['id'],
        'RFM': rfm,
        'Arquetipo': arquetipo,
        'Journey': journey,
        'Afinidad': afinidad,
        'Canal': canal,
        'Acción': action[:55] + '...' if len(action) > 55 else action
    })

df = pd.DataFrame(tabla_reglas)
print(df.to_string(index=False))

print("\n" + "=" * 120)
print(f"\nTotal de nuevas reglas: {len(reglas_nuevas)}")
print(f"Todas utilizan exactamente 5 variables:")
print("  ✓ rfm-monetary (valor del cliente)")
print("  ✓ arquetipo (tipo de cliente)")
print("  ✓ customer-journey (etapa del ciclo de vida)")
print("  ✓ afinidad (producto/servicio)")
print("  ✓ canal (canal de contacto)")
print("\n" + "=" * 120)

# Estadísticas por categoría
print("\n\nESTADÍSTICAS POR CATEGORÍA:\n")

print("Por RFM Monetary:")
for rfm in ['alto', 'medio', 'bajo']:
    count = len([r for r in reglas_nuevas if any(f'rfm-monetary-{rfm}' in c for c in r['si'])])
    print(f"  {rfm.upper():8} → {count} reglas")

print("\nPor Arquetipo:")
arquetipos = ['cliente_patrimonial', 'profesional_joven_digital', 'familia_en_expansion', 'cliente_transaccional', 'emprendedor_pequeño_empresario']
for arq in arquetipos:
    count = len([r for r in reglas_nuevas if any(f'arquetipo_{arq}' in c for c in r['si'])])
    print(f"  {arq:35} → {count} reglas")

print("\nPor Journey:")
journeys = ['adquisicion', 'activacion', 'crecimiento', 'madurez', 'reactivacion', 'riesgo_de_abandono']
for j in journeys:
    count = len([r for r in reglas_nuevas if any(f'customer-journey-{j}' in c for c in r['si'])])
    if count > 0:
        print(f"  {j:20} → {count} reglas")

print("\nPor Afinidad:")
afinidades = ['fondos_inversion', 'tarjeta_credito', 'seguros', 'cuenta_corriente', 'credito_personal']
for af in afinidades:
    count = len([r for r in reglas_nuevas if any(f'afinidad_{af}' in c for c in r['si'])])
    print(f"  {af:15} → {count} reglas")

print("\nPor Canal:")
canales = ['asesor_personal', 'aplicacion_movil', 'email', 'sms', 'telefono', 'sucursal']
for c in canales:
    count = len([r for r in reglas_nuevas if any(f'canal_{c}' in cr for cr in r['si'])])
    if count > 0:
        print(f"  {c:18} → {count} reglas")

print("\n" + "=" * 120)



LISTADO COMPLETO DE NUEVAS REGLAS (R46-R65): 5 VARIABLES MÍNIMAS



NameError: name 'reglas_nuevas' is not defined

## Sistema CRM con 65 Reglas: Comparación Original vs Extendido

Vista comparativa del sistema original (45 reglas) con el extendido (65 reglas).

In [ ]:
# ============================================================================
# COMPARACIÓN: Sistema Original vs Sistema Extendido
# ============================================================================

print("\n" + "=" * 100)
print("COMPARACIÓN: SISTEMA ORIGINAL (45 reglas) vs SISTEMA EXTENDIDO (65 reglas)")
print("=" * 100 + "\n")

comparacion_datos = {
    'Característica': [
        'Total de Reglas',
        'Reglas con 5 variables',
        'Reglas con 6 variables',
        'Reglas con 7 variables',
        'Variables mínimas requeridas',
        'Arquetipos soportados',
        'Etapas de journey',
        'Productos/Afinidades',
        'Canales de contacto',
        'Sistema de uso'
    ],
    'Original (Sistema CRM)': [
        f'{len(reglas_originales)}',
        f'{len([r for r in reglas_originales if len(r["si"]) == 5])}',
        f'{len([r for r in reglas_originales if len(r["si"]) == 6])}',
        f'{len([r for r in reglas_originales if len(r["si"]) == 7])}',
        '5 (RFM + Arquetipo + Journey + Afinidad + Canal)',
        '5',
        '6',
        '5',
        '6',
        'sistema'
    ],
    'Extendido (SistemaCRMExtendido)': [
        f'{len(TODAS_LAS_REGLAS)}',
        f'{len([r for r in TODAS_LAS_REGLAS if len(r["si"]) == 5])}',
        f'{len([r for r in TODAS_LAS_REGLAS if len(r["si"]) == 6])}',
        f'{len([r for r in TODAS_LAS_REGLAS if len(r["si"]) == 7])}',
        '5 (RFM + Arquetipo + Journey + Afinidad + Canal)',
        '5',
        '6',
        '5',
        '6',
        'sistema_extendido'
    ]
}

df_comp = pd.DataFrame(comparacion_datos)
print(df_comp.to_string(index=False))

print("\n" + "=" * 100)
print("\n✓ VENTAJAS DEL SISTEMA EXTENDIDO:")
print("  • 20 nuevas reglas adicionales con combinaciones optimizadas")
print("  • Todas las nuevas reglas usan exactamente 5 variables (especificidad media)")
print("  • Mayor cobertura de escenarios: 65 reglas vs 45 reglas")
print("  • Compatible con todas las dimensiones CRM: RFM + Arquetipos + Journey + Afinidad + Canal")
print("  • Mejor resolución de conflictos gracias a reglas de diferente especificidad")

print("\n✓ CÓMO USAR EL SISTEMA EXTENDIDO:")
print("  1. Usar 'sistema' para aplicar solo las 45 reglas originales")
print("  2. Usar 'sistema_extendido' para aplicar las 65 reglas completas")
print("  3. Ambos sistemas heredan de SistemaCRM y usan el mismo motor de inferencia")
print("  4. Las reglas más específicas (6-7 variables) tienen mayor prioridad")

print("\n" + "=" * 100)
